<a href="https://colab.research.google.com/github/Trangnguyen1402/AAI2025/blob/main/Inventory_Replenishment_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Create Synthetic Sales Data

Because a real sales dataset was not provided, this project uses synthetic data generated with Python. The dataset contains 90 days of daily demand for three SKUs. Each SKU has a different average demand level, and random variation is included to make the data more realistic for forecasting.

In [2]:
# Generate sales.csv

import numpy as np
import pandas as pd

# Set a seed so the same data is created every time
rng = np.random.default_rng(7)

# Create 90 days of sales dates
dates = pd.date_range(
    start="2025-01-01",
    periods=90,
    freq="D"
)

# Use three different products
skus = ["SKU001", "SKU002", "SKU003"]

# Average daily demand for each SKU
average_demand = {
    "SKU001": 20,
    "SKU002": 12,
    "SKU003": 8
}

sales_rows = []

for sku in skus:
    for date in dates:

        # Weekly demand pattern
        weekly_effect = 3 * np.sin(
            2 * np.pi * date.dayofweek / 7
        )

        # Random demand variation
        random_effect = rng.normal(0, 3)

        # Make sure demand is never negative
        qty_sold = max(
            0,
            round(
                average_demand[sku]
                + weekly_effect
                + random_effect
            )
        )

        sales_rows.append({
            "date": date,
            "sku": sku,
            "qty_sold": qty_sold
        })

# Convert the list into a DataFrame
sales = pd.DataFrame(sales_rows)

# Save the data as a CSV file
sales.to_csv("sales.csv", index=False)

# Display the first few rows
print("sales.csv was created successfully.")
display(sales.head(10))

sales.csv was created successfully.


,date,sku,qty_sold
0,2025-01-01,SKU001,23
1,2025-01-02,SKU001,22
2,2025-01-03,SKU001,18
3,2025-01-04,SKU001,14
4,2025-01-05,SKU001,16
5,2025-01-06,SKU001,17
6,2025-01-07,SKU001,23
7,2025-01-08,SKU001,27
8,2025-01-09,SKU001,20
9,2025-01-10,SKU001,17


## Explanation of the Sales Data

The sales dataset contains three columns:

- `date`: The date when the product was sold.
- `sku`: The product identification number.
- `qty_sold`: The number of units sold on that date.

The data includes 90 days of sales for three products. SKU001 has the highest average demand, SKU002 has medium demand, and SKU003 has the lowest demand. A weekly demand pattern and random variation were added so the data can be used to test the EWMA forecasting model and inventory replenishment agent.

In [3]:
# Check the size and structure of the sales data

print("Number of rows:", len(sales))
print("Number of SKUs:", sales["sku"].nunique())
print("Date range:", sales["date"].min(), "to", sales["date"].max())

print("\nSales by SKU:")
display(sales.groupby("sku")["qty_sold"].agg(["count", "mean", "sum"]))

print("\nMissing values:")
display(sales.isnull().sum())

Number of rows: 270
Number of SKUs: 3
Date range: 2025-01-01 00:00:00 to 2025-03-31 00:00:00

Sales by SKU:


,count,mean,sum
sku,,,
SKU001,90,19.500000,1755
SKU002,90,11.377778,1024
SKU003,90,7.666667,690



Missing values:


,0
date,0
sku,0
qty_sold,0


In [4]:
# Save the DataFrame as a CSV file
sales.to_csv("sales.csv", index=False)

# Download the CSV file to your computer
from google.colab import files
files.download("sales.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Create Inventory and Policy Parameter Data

The inventory file contains the opening stock available for each SKU at the beginning of the simulation. The parameter file contains the costs and replenishment rules used by the inventory agent.

Each SKU has different starting inventory, lead time, minimum order quantity, and service-level requirements. These values allow the agent to make realistic order-or-wait decisions.

In [6]:
# Create inventory.csv

inventory = pd.DataFrame({
    "sku": ["SKU001", "SKU002", "SKU003"],
    "opening_stock": [180, 120, 80]
})

# Save the file
inventory.to_csv("inventory.csv", index=False)

print("inventory.csv was created successfully.")
display(inventory)

inventory.csv was created successfully.


,sku,opening_stock
0,SKU001,180
1,SKU002,120
2,SKU003,80


In [7]:
# Create params.csv

params = pd.DataFrame({
    "sku": ["SKU001", "SKU002", "SKU003"],
    "unit_cost": [10.00, 15.00, 20.00],
    "holding_cost_per_day": [0.20, 0.30, 0.40],
    "stockout_cost": [25.00, 30.00, 40.00],
    "lead_time_days": [5, 7, 4],
    "min_order_qty": [50, 30, 20],
    "service_level": [0.95, 0.90, 0.95]
})

# Save the file
params.to_csv("params.csv", index=False)

print("params.csv was created successfully.")
display(params)

params.csv was created successfully.


,sku,unit_cost,holding_cost_per_day,stockout_cost,lead_time_days,min_order_qty,service_level
0,SKU001,10.0,0.2,25.0,5,50,0.95
1,SKU002,15.0,0.3,30.0,7,30,0.90
2,SKU003,20.0,0.4,40.0,4,20,0.95


## Explanation of Inventory and Policy Parameters

The `inventory.csv` file contains the opening stock for each product. SKU001 starts with 180 units, SKU002 starts with 120 units, and SKU003 starts with 80 units.

The `params.csv` file contains the business rules and cost information for each SKU:

- `unit_cost`: Cost of purchasing one unit.
- `holding_cost_per_day`: Cost of storing one unit for one day.
- `stockout_cost`: Penalty for each unit of unmet demand.
- `lead_time_days`: Number of days between placing and receiving an order.
- `min_order_qty`: Smallest quantity the agent is allowed to order.
- `service_level`: Target probability of meeting customer demand.

The agent uses these values to balance customer service and inventory cost. A higher service level and stockout cost encourage the agent to keep more safety stock, while holding cost discourages unnecessary over-ordering.

In [8]:
# Verify that all required CSV files exist and have the correct columns

sales_check = pd.read_csv("sales.csv")
inventory_check = pd.read_csv("inventory.csv")
params_check = pd.read_csv("params.csv")

assert list(sales_check.columns) == ["date", "sku", "qty_sold"]

assert list(inventory_check.columns) == [
    "sku",
    "opening_stock"
]

assert list(params_check.columns) == [
    "sku",
    "unit_cost",
    "holding_cost_per_day",
    "stockout_cost",
    "lead_time_days",
    "min_order_qty",
    "service_level"
]

assert set(sales_check["sku"]) == set(inventory_check["sku"])
assert set(sales_check["sku"]) == set(params_check["sku"])

print("All three CSV files have the required format.")

All three CSV files have the required format.


In [9]:
from google.colab import files

files.download("inventory.csv")
files.download("params.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>